# Plot From A Batch Case


This notebook loads a completed case from a batch run using only the batch run name and the case key.


It reads the batch summary, resolves the corresponding result directory, loads the original and permuted matrices, and then displays the plots you request for that case.

In [ ]:
import gc
import matplotlib.pyplot as plt
from pathlib import Path
import traceback
import importlib
import notebook_functions

importlib.reload(notebook_functions)
from notebook_functions import *

ROOT_OUTPUT = Path("results_images")
ROOT_OUTPUT.mkdir(exist_ok=True, parents=True)

BATCH_RUN_NAME = "batch_20260521_151902"
completed_rows = get_completed_rows(BATCH_RUN_NAME)
DEMONSTRATORS = ["demonstrator_02"]

print(f"Processing demonstrators: {DEMONSTRATORS}")

METHODS = ["qaoa_hardware", "qaoa_emulated", "qa_hardware", "qa_emulated", "metis"]

PLOT_SPEC = [
    ("k_original_matrix_partitioned", True),
    ("k_permuted_matrix_partitioned", False),
    ("k_original_graph_partitioned", True),
    ("k_coarsened_graph_partitioned", False),
    ("k_uncoarsened_graph_partitioned", False),
]

def find_row_for(case_key):
    for r in completed_rows:
        if r.get("case_key") == case_key:
            return r
    return None

created_files = []

for dem in DEMONSTRATORS:
    dem_short = "dem" + "".join([c for c in dem if c.isdigit()])
    for method in METHODS:
        sizes = sorted({r["case_key"].split("__")[-1] for r in completed_rows if r["case_key"].startswith(f"{dem}__{method}__")})
        if not sizes:
            print(f"No completed cases for {dem} / {method}. Skipping.")
            continue
        for size in sizes:
            case_key = f"{dem}__{method}__{size}"
            row = find_row_for(case_key)
            if row is None:
                print(f"Case {case_key} not found in completed_rows. Skipping.")
                continue
            try:
                case = load_case_from_summary(row)
            except Exception as e:
                print(f"Failed to load case {case_key}: {e}")
                traceback.print_exc()
                continue
            out_dir = ROOT_OUTPUT / dem_short / method / f"coarsen_{size}"
            out_dir.mkdir(parents=True, exist_ok=True)
            for plot_name, show_y in PLOT_SPEC:
                out_file = out_dir / f"{plot_name}.png"
                try:
                    render_case_plot(case, method, plot_name, display_width=700, display_height=700, title_fontsize=16, label_fontsize=14, tick_labelsize=12, show_y_axis=show_y, save_path=str(out_file))
                    created_files.append(out_file)
                    print(f"Saved {out_file}")
                except Exception as e:
                    print(f"Failed to render {plot_name} for {case_key}: {e}")
                    traceback.print_exc()
                finally:
                    plt.close("all")

            if method != "metis":
                metis_key = f"{dem}__metis__{size}"
                metis_row = find_row_for(metis_key)
                if metis_row is None:
                    print(f"No metis counterpart for {case_key} (expected {metis_key}). Skipping curve.")
                else:
                    metis_case = None
                    try:
                        metis_case = load_case_from_summary(metis_row)
                        if case.get("run_metrics") is None or metis_case.get("run_metrics") is None:
                            print(f"Missing run_metrics for curve comparison {case_key} vs {metis_key}. Skipping curve.")
                        else:
                            out_curve = out_dir / f"k_curve_comparison_{method}_vs_metis.png"
                            render_k_curve_comparison(case, metis_case, title=f"{NAME_MAPPING.get(method, method)} {NAME_MAPPING.get(size,size)} vs Metis {NAME_MAPPING.get(size,size)}", backend_label=f"{NAME_MAPPING.get(method,method)} {NAME_MAPPING.get(size,size)}", metis_label=f"Metis {NAME_MAPPING.get(size,size)}", display_width=1200, display_height=700, title_fontsize=16, label_fontsize=14, tick_labelsize=12, save_path=str(out_curve))
                            created_files.append(out_curve)
                            print(f"Saved {out_curve}")
                    except Exception as e:
                        print(f"Failed curve comparison for {case_key} vs {metis_key}: {e}")
                        traceback.print_exc()
                    finally:
                        plt.close("all")
                        del metis_case
                        metis_case = None

            # Free case and force cleanup
            del case
            case = None
            plt.close("all")
            gc.collect()

    gc.collect()

print(f"Done. Created {len(created_files)} images.")

Batch directory: /home/operation/Thesis/hybrid_domain_decomposition/batch/runs/batch_20260521_151902
Cases in summary: 20
Completed cases with result_dir: 18

Available cases:
  - demonstrator_01__metis__big (status: completed)
  - demonstrator_01__metis__medium (status: completed)
  - demonstrator_01__metis__small (status: completed)
  - demonstrator_01__qa_hardware__medium (status: completed)
  - demonstrator_01__qa_hardware__small (status: completed)
  - demonstrator_01__qa_emulated__big (status: completed)
  - demonstrator_01__qa_emulated__medium (status: completed)
  - demonstrator_01__qa_emulated__small (status: completed)
  - demonstrator_01__qaoa_hardware__small (status: failed)
  - demonstrator_01__qaoa_emulated__small (status: completed)
  - demonstrator_02__metis__big (status: completed)
  - demonstrator_02__metis__medium (status: completed)
  - demonstrator_02__metis__small (status: completed)
  - demonstrator_02__qa_hardware__medium (status: completed)
  - demonstrator_02_

In [10]:
import importlib
import json
from pathlib import Path

import notebook_functions

importlib.reload(notebook_functions)
from notebook_functions import *

BATCH_RUN_NAME = "batch_20260521_151902"
DEMONSTRATOR = "demonstrator_01"

repo_root = Path(REPO_ROOT) if "REPO_ROOT" in globals() else Path.cwd()
summary_path = repo_root / "batch" / "runs" / BATCH_RUN_NAME / "batch_summary.json"
if not summary_path.exists():
    print(f"Batch summary not found: {summary_path}")
else:
    with summary_path.open() as fh:
        batch_summary = json.load(fh)

    rows = batch_summary.get("rows", [])
    demo_rows = [
        row
        for row in rows
        if row.get("status") == "completed"
        and row.get("case_key", "").startswith(f"{DEMONSTRATOR}__")
    ]

    if not demo_rows:
        print(f"No completed rows found for {DEMONSTRATOR} in {BATCH_RUN_NAME}")
    else:
        row_by_case = {row["case_key"]: row for row in demo_rows}
        original_row = demo_rows[0]

        def fmt_int(value):
            return str(int(round(float(value)))) if value is not None else "N/A"

        def fmt_avg(value):
            return f"{float(value):.2f}" if value is not None else "N/A"

        def fmt_frac(value):
            return f"{float(value):.4f}" if value is not None else "N/A"

        def pretty_method(method: str) -> str:
            return {
                "metis": "METIS",
                "qa_hardware": "QA hardware",
                "qa_emulated": "QA emulated",
                "qaoa_emulated": "QAOA emulated",
                "qaoa_hardware": "QAOA hardware",
            }.get(method, method)

        method_order = ["metis", "qa_hardware", "qa_emulated", "qaoa_emulated", "qaoa_hardware"]
        size_order = ["big", "medium", "small"]
        row_end = chr(92) * 2

        table_rows = [
            (
                "Original",
                "N/A",
                fmt_int(original_row.get("K_bandwidth_original")),
                fmt_avg(original_row.get("K_avg_bandwidth_original")),
                fmt_frac(original_row.get("K_frac_0_1pct_original")),
                fmt_frac(original_row.get("K_frac_0_4pct_original")),
                fmt_frac(original_row.get("K_frac_0_7pct_original")),
                fmt_frac(original_row.get("K_frac_1_0pct_original")),
            )
        ]

        for method in method_order:
            for size in size_order:
                case_key = f"{DEMONSTRATOR}__{method}__{size}"
                row = row_by_case.get(case_key)
                if row is None:
                    continue
                table_rows.append(
                    (
                        pretty_method(method),
                        size,
                        fmt_int(row.get("K_bandwidth_permuted")),
                        fmt_avg(row.get("K_avg_bandwidth_permuted")),
                        fmt_frac(row.get("K_frac_0_1pct_permuted")),
                        fmt_frac(row.get("K_frac_0_4pct_permuted")),
                        fmt_frac(row.get("K_frac_0_7pct_permuted")),
                        fmt_frac(row.get("K_frac_1_0pct_permuted")),
                    )
                )

        demo_suffix = "".join(ch for ch in DEMONSTRATOR if ch.isdigit())
        demo_display = f"Demonstrator~{demo_suffix}" if demo_suffix else DEMONSTRATOR
        demo_label = f"tab:results_demo{int(demo_suffix)}_k" if demo_suffix else "tab:results_k"

        lines = [
            "\\begin{table}[H]",
            f"\\caption{{K-matrix numerical summary for {demo_display}}}",
            "\\scriptsize",
            "\\renewcommand{\\arraystretch}{1.15}",
            "\\begin{tabularx}{\\textwidth}{X l r r r r r r}",
            "\\toprule",
            "\\textbf{Method} & \\textbf{Size} & \\textbf{Bandwidth} & \\textbf{Avg. Bandwidth} & \\textbf{$\\mathrm{frac}_{0.1\\%}$} & \\textbf{$\\mathrm{frac}_{0.4\\%}$} & \\textbf{$\\mathrm{frac}_{0.7\\%}$} & \\textbf{$\\mathrm{frac}_{1.0\\%}$}" + row_end,
            "\\midrule",
        ]

        for method_name, size_name, bandwidth, avg_bandwidth, frac_01, frac_04, frac_07, frac_10 in table_rows:
            lines.append(
                f"{method_name} & {size_name} & {bandwidth} & {avg_bandwidth} & {frac_01} & {frac_04} & {frac_07} & {frac_10}" + row_end
            )

        lines.extend([
            "\\bottomrule",
            "\\end{tabularx}",
            "\\centering",
            "Source: Author (2026)",
            f"\\label{{{demo_label}}}",
            "\\end{table}",
        ])

        print("\n".join(lines))

\begin{table}[H]
\caption{K-matrix numerical summary for Demonstrator~01}
\scriptsize
\renewcommand{\arraystretch}{1.15}
\begin{tabularx}{\textwidth}{X l r r r r r r}
\toprule
\textbf{Method} & \textbf{Size} & \textbf{Bandwidth} & \textbf{Avg. Bandwidth} & \textbf{$\mathrm{frac}_{0.1\%}$} & \textbf{$\mathrm{frac}_{0.4\%}$} & \textbf{$\mathrm{frac}_{0.7\%}$} & \textbf{$\mathrm{frac}_{1.0\%}$}\\
\midrule
Original & N/A & 309635 & 2695.46 & 0.2926 & 0.6724 & 0.8515 & 0.9234\\
METIS & big & 251021 & 7413.11 & 0.3318 & 0.7056 & 0.8483 & 0.9019\\
METIS & medium & 310781 & 6404.33 & 0.3386 & 0.7071 & 0.8529 & 0.9087\\
METIS & small & 258956 & 4994.66 & 0.3848 & 0.7795 & 0.8917 & 0.9289\\
QA hardware & medium & 311540 & 6661.11 & 0.3404 & 0.7237 & 0.8633 & 0.9155\\
QA hardware & small & 306515 & 5680.84 & 0.3630 & 0.7500 & 0.8756 & 0.9201\\
QA emulated & big & 250511 & 3770.20 & 0.3070 & 0.6815 & 0.8484 & 0.9137\\
QA emulated & medium & 283400 & 5610.28 & 0.3341 & 0.6998 & 0.8482 & 0.9047\\
QA